# Polyp Detection — Final Evaluation

Loads saved metrics from all experiments and builds the final
comparison tables and figures. No new training or inference here —
this notebook only reads from results/metrics/.

---

## What this notebook does

1. Loads metrics from notebooks 03, 04, 04b, 04c, 04d
2. Builds a side-by-side comparison table across all approaches
3. Generates figures for the README
4. Prints README-ready result text

## Models compared

- Baseline YOLOv8m-seg (imgsz=640, conf=0.50)
- Threshold tuning (conf=0.30, same weights)
- Higher resolution (imgsz=960, failed experiment)
- Aug + Negatives — final model

## Output

- results/figures/final_comparison.png
- results/figures/recall_progression.png
- results/figures/error_analysis_summary.png
- results/metrics/final_summary.json

In [ ]:
# Import libraries

import json
import os
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "baseline_metrics":    os.path.join(BASE_DIR, "results", "metrics", "baseline_metrics.json"),
    "safety_first_metrics": os.path.join(BASE_DIR, "results", "metrics", "safety_first_metrics.json"),
    "imgsz960_metrics":    os.path.join(BASE_DIR, "results", "metrics", "imgsz960_metrics.json"),
    "aug_neg_metrics":     os.path.join(BASE_DIR, "results", "metrics", "aug_neg_metrics.json"),
    "error_analysis":      os.path.join(BASE_DIR, "results", "metrics", "error_analysis_summary.json"),
    "figures":             os.path.join(BASE_DIR, "results", "figures"),
    "metrics":             os.path.join(BASE_DIR, "results", "metrics"),
}

os.makedirs(PATHS["figures"], exist_ok=True)
os.makedirs(PATHS["metrics"], exist_ok=True)

print("Checking metric files:")
for name, path in PATHS.items():
    if name in ("figures", "metrics"):
        continue
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {name}")

In [ ]:
# Load All Metrics
# Reads JSON files saved by notebooks 03, 04, 04b, 04c, 04d

def load_json(path):
    with open(path) as f:
        return json.load(f)


baseline    = load_json(PATHS["baseline_metrics"])
threshold   = load_json(PATHS["safety_first_metrics"])
imgsz960    = load_json(PATHS["imgsz960_metrics"])
aug_neg     = load_json(PATHS["aug_neg_metrics"])
error_stats = load_json(PATHS["error_analysis"])

print("All metrics loaded successfully.")
print()
print("Baseline CVC recall:    ", baseline["cvc_cross_dataset"]["recall"])
print("Threshold CVC recall:   ", threshold["cvc_cross_dataset"]["recall"])
print("imgsz960 CVC recall:    ", imgsz960["imgsz_960"]["cvc_recall"])
print("Aug+Neg CVC recall:     ", aug_neg["cvc_cross_dataset"]["recall"])

In [ ]:
# Build Comparison Table
# Structured summary of all approaches for easy printing
# and for the README table

approaches = [
    {
        "name":          "Baseline (conf=0.50)",
        "kvasir_recall": baseline["kvasir_test"]["recall"],
        "kvasir_prec":   baseline["kvasir_test"]["precision"],
        "cvc_recall":    baseline["cvc_cross_dataset"]["recall"],
        "cvc_prec":      baseline["cvc_cross_dataset"]["precision"],
        "notes":         "Standard YOLOv8m-seg, default settings",
    },
    {
        "name":          "Threshold Tuning (conf=0.30)",
        "kvasir_recall": threshold["kvasir_test"]["recall"],
        "kvasir_prec":   threshold["kvasir_test"]["precision"],
        "cvc_recall":    threshold["cvc_cross_dataset"]["recall"],
        "cvc_prec":      threshold["cvc_cross_dataset"]["precision"],
        "notes":         "Same model, lower confidence threshold",
    },
    {
        "name":          "Higher Resolution (imgsz=960)",
        "kvasir_recall": imgsz960["imgsz_960"]["kvasir_recall"],
        "kvasir_prec":   imgsz960["imgsz_960"]["kvasir_precision"],
        "cvc_recall":    imgsz960["imgsz_960"]["cvc_recall"],
        "cvc_prec":      imgsz960["imgsz_960"]["cvc_precision"],
        "notes":         "Retrain at 960px - batch=4, did not improve",
    },
    {
        "name":          "Aug + Negatives (final)",
        "kvasir_recall": aug_neg["kvasir_test"]["recall"],
        "kvasir_prec":   aug_neg["kvasir_test"]["precision"],
        "cvc_recall":    aug_neg["cvc_cross_dataset"]["recall"],
        "cvc_prec":      aug_neg["cvc_cross_dataset"]["precision"],
        "notes":         "Stronger augmentation + 140 negative frames",
    },
]

print("=" * 90)
print(f"{'Approach':<35} {'Kvasir R':>9} {'Kvasir P':>9} {'CVC R':>9} {'CVC P':>9}")
print("-" * 90)
for a in approaches:
    marker = " <-- BEST" if a["name"].startswith("Aug") else ""
    print(f"{a['name']:<35} "
          f"{a['kvasir_recall']:>9.3f} "
          f"{a['kvasir_prec']:>9.3f} "
          f"{a['cvc_recall']:>9.3f} "
          f"{a['cvc_prec']:>9.3f}"
          f"{marker}")
print("=" * 90)

In [ ]:
# Main Comparison Chart
# Bar chart comparing recall across all approaches
# on both datasets - this is the core README figure

labels    = [a["name"] for a in approaches]
kv_recall = [a["kvasir_recall"] for a in approaches]
cv_recall = [a["cvc_recall"] for a in approaches]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))

bars1 = ax.bar(x - width/2, kv_recall, width,
               label="Kvasir-SEG (in-distribution)",
               color="steelblue", edgecolor="white")
bars2 = ax.bar(x + width/2, cv_recall, width,
               label="CVC-ClinicDB (cross-dataset)",
               color="coral", edgecolor="white")

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Recall")
ax.set_title("Recall Comparison Across All Approaches",
             fontsize=13, fontweight="bold")
ax.set_ylim(0.65, 1.0)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "final_comparison.png")
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Recall Progression (Story Chart)
# Line chart showing how CVC recall improved step by step

steps = [
    "Baseline\n(conf=0.50)",
    "Threshold\nTuning",
    "imgsz=960\n(failed)",
    "Aug +\nNegatives",
]
cvc_progression = [
    baseline["cvc_cross_dataset"]["recall"],
    threshold["cvc_cross_dataset"]["recall"],
    imgsz960["imgsz_960"]["cvc_recall"],
    aug_neg["cvc_cross_dataset"]["recall"],
]

colors = ["steelblue", "steelblue", "tomato", "seagreen"]

fig, ax = plt.subplots(figsize=(10, 5))

for i in range(len(steps) - 1):
    ax.plot([i, i+1], [cvc_progression[i], cvc_progression[i+1]],
            color="gray", linewidth=1.5, linestyle="--", zorder=1)

for i, (step, val, color) in enumerate(zip(steps, cvc_progression, colors)):
    ax.scatter(i, val, color=color, s=120, zorder=2)
    ax.text(i, val + 0.006, f"{val:.3f}", ha="center", fontsize=10,
            fontweight="bold", color=color)

ax.axhline(cvc_progression[0], color="steelblue", linestyle=":",
           linewidth=1, alpha=0.5, label=f"Baseline ({cvc_progression[0]:.3f})")

ax.set_xticks(range(len(steps)))
ax.set_xticklabels(steps, fontsize=10)
ax.set_ylabel("CVC-ClinicDB Recall (cross-dataset)")
ax.set_title("Cross-Dataset Recall Improvement Journey",
             fontsize=13, fontweight="bold")
ax.set_ylim(0.68, 0.88)

legend_handles = [
    mpatches.Patch(color="steelblue", label="Neutral / starting point"),
    mpatches.Patch(color="tomato",    label="Failed experiment"),
    mpatches.Patch(color="seagreen",  label="Successful improvement"),
]
ax.legend(handles=legend_handles, fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "recall_progression.png")
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Error Analysis Summary Chart
# Visualize the size gap finding from notebook 04b

categories    = ["Detected", "Missed"]
median_areas  = [
    error_stats["median_area_detected"] * 100,
    error_stats["median_area_missed"]   * 100,
]

fig, ax = plt.subplots(figsize=(6, 5))

bars = ax.bar(categories, median_areas,
              color=["steelblue", "coral"],
              edgecolor="white", width=0.4)

for bar, val in zip(bars, median_areas):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1,
            f"{val:.2f}%", ha="center", fontweight="bold")

ax.set_ylabel("Median polyp area (% of image)")
ax.set_title("Why Polyps Were Missed:\nDetected vs Missed Size Comparison",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 15)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "error_analysis_summary.png")
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Save Final Summary JSON
# Single clean summary of the final model for README and Streamlit app

final_summary = {
    "final_model": {
        "weights":   "models/aug_neg/weights/best.pt",
        "config":    "YOLOv8m-seg, imgsz=640, conf=0.30",
        "training":  "aug+neg (flipud=0.5, degrees=45, 140 negative frames)",
    },
    "results": {
        "kvasir_test": {
            "recall":    aug_neg["kvasir_test"]["recall"],
            "precision": aug_neg["kvasir_test"]["precision"],
            "map50":     aug_neg["kvasir_test"]["map50"],
        },
        "cvc_cross_dataset": {
            "recall":    aug_neg["cvc_cross_dataset"]["recall"],
            "precision": aug_neg["cvc_cross_dataset"]["precision"],
            "map50":     aug_neg["cvc_cross_dataset"]["map50"],
        },
    },
    "baseline_comparison": {
        "cvc_recall_baseline":  baseline["cvc_cross_dataset"]["recall"],
        "cvc_recall_final":     aug_neg["cvc_cross_dataset"]["recall"],
        "improvement":          aug_neg["cvc_cross_dataset"]["recall"] - baseline["cvc_cross_dataset"]["recall"],
    },
    "inference_speed": {
        "ms_per_image": baseline["inference_speed_ms"],
        "fps":          baseline["inference_fps"],
    },
}

summary_path = os.path.join(PATHS["metrics"], "final_summary.json")
with open(summary_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print(f"Saved -> {summary_path}")
print()
print(json.dumps(final_summary, indent=2))

In [ ]:
# README-Ready Text
# Print the key numbers formatted for direct copy-paste
# into the README results section

print("=" * 60)
print("README-READY RESULTS")
print("=" * 60)

improvement = final_summary["baseline_comparison"]["improvement"]

print(f"""
## Key Results

| Metric | Baseline | Final Model | Change |
|--------|----------|-------------|--------|
| Kvasir-SEG Recall | {baseline['kvasir_test']['recall']:.3f} | {aug_neg['kvasir_test']['recall']:.3f} | +{aug_neg['kvasir_test']['recall'] - baseline['kvasir_test']['recall']:.3f} |
| CVC-ClinicDB Recall | {baseline['cvc_cross_dataset']['recall']:.3f} | {aug_neg['cvc_cross_dataset']['recall']:.3f} | +{improvement:.3f} |
| Inference Speed | {baseline['inference_fps']:.0f} FPS | {baseline['inference_fps']:.0f} FPS | same |

### Cross-Dataset Generalization
The final model achieves **{aug_neg['cvc_cross_dataset']['recall']:.1%} recall** on CVC-ClinicDB,
a dataset never seen during training (+{improvement:.1%} over baseline).

### Engineering Journey
1. Identified root cause: missed polyps were 59% smaller (median) than detected ones
2. Tested imgsz=960 → no improvement (batch reduction hurt training stability)
3. Applied augmentation + negative samples (per YOLO-LAN 2025) → +6.2% CVC recall
""")

In [ ]:
# Summary

print("NOTEBOOK 05 COMPLETE")
print("=" * 50)
print("Figures saved:")
for fname in ["final_comparison.png", "recall_progression.png",
              "error_analysis_summary.png"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Final model: models/aug_neg/weights/best.pt")
print(f"CVC Recall:  {aug_neg['cvc_cross_dataset']['recall']:.3f}")
print()
print("Next -> 06_video_tracking.ipynb")
print("  - Apply final model on a real colonoscopy video")
print("  - Add ByteTrack for polyp tracking across frames")
print("  - Generate demo video for LinkedIn post")